In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [ ]:
from pathlib import Path
import yaml

# ============================================================
# VisDrone_Final — Dataset YAML
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final"
)

YAML_PATH = Path("/kaggle/working/visdrone_final.yaml")

# Check dataset
print("Dataset exists:", DATASET_ROOT.exists())
print("Dataset path  :", DATASET_ROOT)

# Check required directories
required_dirs = [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "train" / "labels",
    DATASET_ROOT / "val" / "images",
    DATASET_ROOT / "val" / "labels",
    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "test" / "labels",
]

for d in required_dirs:
    print(f"{d}: {d.exists()}")

# ============================================================
# YAML configuration
# ============================================================

data = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",

    "nc": 10,

    "names": [
        "pedestrian",
        "people",
        "bicycle",
        "car",
        "van",
        "truck",
        "tricycle",
        "awning-tricycle",
        "bus",
        "motor"
    ]
}

# Write YAML
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("\n" + "=" * 60)
print("YAML CREATED")
print("=" * 60)

print(YAML_PATH)
print("\nContents:\n")

with open(YAML_PATH, "r") as f:
    print(f.read())

In [ ]:
# ============================================================
# YOLOv8n + Micro-SKA
# CLEAN, SELF-CONTAINED EXPERIMENT
# ============================================================

import os
import json
import math
import random
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from ultralytics import YOLO

print("=" * 70)
print("YOLOv8n + Micro-SKA")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# BUILD 10-CLASS YOLOv8n FROM ARCHITECTURE YAML
# AND TRANSFER COCO PRETRAINED WEIGHTS
# ============================================================

import os
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = "/kaggle/working/visdrone_final.yaml"
PROJECT_DIR = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLOv8n_MicroSKA_AdamW_seed42"

assert os.path.exists(DATA_YAML), \
    f"Dataset YAML not found: {DATA_YAML}"

print("=" * 70)
print("BUILDING 10-CLASS YOLOv8n")
print("=" * 70)


# ------------------------------------------------------------
# 1. Load standard YOLOv8n architecture
# ------------------------------------------------------------

print("\nLoading standard YOLOv8n architecture...")

base_80 = YOLO("yolov8n.yaml")

print("✓ Standard YOLOv8n architecture loaded")


# ------------------------------------------------------------
# 2. Extract architecture YAML
# ------------------------------------------------------------

model_yaml = dict(base_80.model.yaml)

print("\nOriginal architecture classes:")
print("nc =", model_yaml["nc"])


# ------------------------------------------------------------
# 3. Change ONLY the number of classes
# ------------------------------------------------------------

model_yaml["nc"] = 10

# Optional class names for clarity
model_yaml["names"] = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor"
]


# ------------------------------------------------------------
# 4. Save 10-class architecture YAML
# ------------------------------------------------------------

CUSTOM_YAML = "/kaggle/working/yolov8n_visdrone_10class.yaml"

with open(CUSTOM_YAML, "w") as f:
    yaml.safe_dump(
        model_yaml,
        f,
        sort_keys=False
    )

print("\n10-class YAML saved:")
print(CUSTOM_YAML)


# ------------------------------------------------------------
# 5. Construct fresh 10-class YOLOv8n
# ------------------------------------------------------------

model = YOLO(CUSTOM_YAML)

params_10class = sum(
    p.numel()
    for p in model.model.parameters()
)

print("\nFresh 10-class YOLOv8n:")
print(f"Parameters: {params_10class:,}")


# ------------------------------------------------------------
# 6. Verify correct architecture
# ------------------------------------------------------------

assert params_10class == 3_012_798, (
    f"Unexpected 10-class parameter count: "
    f"{params_10class:,}"
)

print("✓ Correct 10-class YOLOv8n confirmed")


# ------------------------------------------------------------
# 7. Load official COCO pretrained model
# ------------------------------------------------------------

print("\nLoading official COCO-pretrained YOLOv8n...")

pretrained_80 = YOLO("yolov8n.pt")

print("✓ COCO pretrained model loaded")

In [ ]:
# ============================================================
# TRANSFER COMPATIBLE COCO PRETRAINED WEIGHTS
# ============================================================

print("=" * 70)
print("TRANSFERRING PRETRAINED WEIGHTS")
print("=" * 70)

source_state = pretrained_80.model.state_dict()
target_state = model.model.state_dict()

transferred = 0
skipped = 0

for key, source_tensor in source_state.items():

    if key not in target_state:
        skipped += 1
        continue

    target_tensor = target_state[key]

    # Only transfer tensors with exactly matching dimensions
    if target_tensor.shape != source_tensor.shape:
        skipped += 1
        continue

    target_tensor.copy_(source_tensor)
    transferred += 1


# Load modified state dictionary
model.model.load_state_dict(
    target_state,
    strict=False
)

print("\nWeight transfer completed")
print(f"Transferred tensors : {transferred}")
print(f"Skipped tensors     : {skipped}")


# ------------------------------------------------------------
# Verify architecture was NOT changed
# ------------------------------------------------------------

params_after_transfer = sum(
    p.numel()
    for p in model.model.parameters()
)

print("\nArchitecture after transfer:")
print(f"Parameters: {params_after_transfer:,}")

assert params_after_transfer == 3_012_798, (
    "Architecture changed during weight transfer"
)

print("\n✓ 10-class architecture preserved")
print("✓ COCO pretrained compatible weights transferred")
print("✓ 10-class detection head retained")

In [ ]:
class MicroSKA(nn.Module):

    def __init__(self, channels, reduction=16):
        super().__init__()

        self.channels = channels

        self.branch3 = nn.Conv2d(
            channels, channels,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=channels,
            bias=False
        )

        self.branch5 = nn.Conv2d(
            channels, channels,
            kernel_size=5,
            stride=1,
            padding=2,
            groups=channels,
            bias=False
        )

        hidden = max(channels // reduction, 8)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
            bias=False
        )

        self.relu = nn.ReLU(inplace=True)

        self.fc2 = nn.Conv2d(
            hidden,
            channels * 2,
            kernel_size=1,
            bias=False
        )

        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):

        x3 = self.branch3(x)
        x5 = self.branch5(x)

        descriptor = self.avg_pool(x)
        descriptor = self.fc1(descriptor)
        descriptor = self.relu(descriptor)
        descriptor = self.fc2(descriptor)

        b = descriptor.shape[0]

        attention = descriptor.view(
            b,
            2,
            self.channels,
            1,
            1
        )

        attention = self.softmax(attention)

        features = torch.stack(
            [x3, x5],
            dim=1
        )

        output = (
            features * attention
        ).sum(dim=1)

        return output

In [ ]:
# ============================================================
# INSERT MICRO-SKA
# ============================================================

TARGET_LAYERS = {
    4: 64,    # P3
    6: 128,   # P4
    8: 256    # P5
}

for layer_idx, channels in TARGET_LAYERS.items():

    original_layer = model.model.model[layer_idx]

    attention = MicroSKA(
        channels=channels,
        reduction=16
    )

    wrapped = nn.Sequential(
        original_layer,
        attention
    )

    wrapped.f = original_layer.f
    wrapped.i = original_layer.i
    wrapped.type = "C2f_MicroSKA"

    model.model.model[layer_idx] = wrapped


# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

micro_ska_count = sum(
    1
    for m in model.model.modules()
    if isinstance(m, MicroSKA)
)

micro_ska_params = sum(
    p.numel()
    for m in model.model.modules()
    if isinstance(m, MicroSKA)
    for p in m.parameters()
)

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

print("=" * 70)
print("MICRO-SKA ARCHITECTURE")
print("=" * 70)

print(f"Micro-SKA modules : {micro_ska_count}")
print(f"Micro-SKA params  : {micro_ska_params:,}")
print(f"Total parameters  : {total_params:,}")

assert micro_ska_count == 3

assert total_params == (
    3_012_798 + micro_ska_params
)

print("\n✓ Three Micro-SKA modules inserted")
print("✓ Parameter count is internally consistent")

In [ ]:
results = model.train(
    data="/kaggle/working/visdrone_final.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,

    optimizer="AdamW",
    lr0=0.002,
    weight_decay=0.0005,

    cos_lr=True,
    warmup_epochs=3,

    amp=True,
    seed=42,
    deterministic=True,

    patience=20,
    max_det=1000,
    workers=2,

    copy_paste=0.0,

    pretrained=False,

    project="/kaggle/working/YOLO_VisDrone_Benchmark",
    name="YOLOv8n_MicroSKA_AdamW_seed42",
    exist_ok=False
)

In [2]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv8n_MicroSKA": "/kaggle/working/YOLO_VisDrone_Benchmark/YOLOv8n_MicroSKA_AdamW_seed42/weights/best.pt",
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n_visdrone_10class summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 2.0±0.7 MB/s, size: 158.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 17.4it/s 1:330.3sss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-correct

In [3]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "YOLOv8n_MicroSKA=/kaggle/working/runs/detect/YOLOv8n_MicroSKA/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: YOLOv8n_MicroSKA
Prediction file: /kaggle/working/runs/detect/YOLOv8n_MicroSKA/predictions.json
Loading and preparing results...
DONE (t=3.63s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=212.15s).
Accumulating evaluation results...
DONE (t=14.68s).
YOLOv8n_MicroSKA all     n= 75102 AP50=0.236 AP50-95=0.131 AR50=0.542 AR50-95=0.295
YOLOv8n_MicroSKA tiny    n= 27346 AP50=0.070 AP50-95=0.025 AR50=0.244 AR50-95=0.095
YOLOv8n_MicroSKA small   n= 23491 AP50=0.187 AP50-95=0.078 AR50=0.548 AR50-95=0.252
YOLOv8n_MicroSKA medium  n= 21856 AP50=0.354 AP50-95=0.199 AR50=0.744 AR50-95=0.428
YOLOv8n_MicroSKA large   n=  2409 AP50=0.469 AP50-95=0.313 AR50=0.847 AR50-95=0.565

Saved:
/kaggle/working/size_results.csv
